In [1]:
!pip install transformers==4.46.3
!pip install rouge-score
!pip install accelerate

In [2]:
import torch
import pandas as pd
import time

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

from rouge_score import rouge_scorer
from tqdm.auto import tqdm

print("Advanced model libraries imported successfully!")

Advanced model libraries imported successfully!


In [3]:
model_name = "sshleifer/distilbart-cnn-12-6"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

# Load pretrained sequence-to-sequence model
bart_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

# Move model to GPU or CPU
bart_model.to(device)

# Evaluation mode
bart_model.eval()

print("Advanced DistilBART model loaded successfully!")

print("Model:", model_name)
print("Device:", device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Advanced DistilBART model loaded successfully!
Model: sshleifer/distilbart-cnn-12-6
Device: cpu


In [4]:
def bart_summary(text):

    text = clean_text(text)

    if not text:
        return ""

    # Tokenization
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=1024,
        truncation=True
    )

    # Move tensors to device
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate summary
    with torch.no_grad():

        summary_ids = bart_model.generate(

            input_ids=inputs["input_ids"],

            attention_mask=inputs["attention_mask"],

            max_new_tokens=100,

            min_new_tokens=15,

            num_beams=4,

            no_repeat_ngram_size=3,

            early_stopping=True,

            do_sample=False
        )

    # Decode generated summary
    summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary


print("DistilBART summarization function created successfully!")

DistilBART summarization function created successfully!


In [8]:
import pandas as pd
import re
from datasets import load_dataset

# Load test dataset
test_data = load_dataset(
    "abisee/cnn_dailymail",
    "3.0.0",
    split="test[:100]"
)

# Clean text
def clean_text(text):
    text = str(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# Prepare test DataFrame
test_df = pd.DataFrame({

    "article": [
        clean_text(item["article"])
        for item in test_data
    ],

    "reference": [
        clean_text(item["highlights"])
        for item in test_data
    ]
})

print("Test dataset loaded successfully!")
print("Shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Test dataset loaded successfully!
Shape: (100, 2)
Columns: ['article', 'reference']


In [9]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

def calculate_rouge(reference, generated):

    scores = scorer.score(
        str(reference),
        str(generated)
    )

    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    }

In [10]:
advanced_results = []

for index, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Evaluating DistilBART"
):

    article = row["article"]
    reference = row["reference"]

    start_time = time.time()

    generated_summary = bart_summary(article)

    execution_time = time.time() - start_time

    scores = calculate_rouge(
        reference,
        generated_summary
    )

    advanced_results.append({

        "Article_ID": index,

        "Reference": reference,

        "BART_Summary": generated_summary,

        "ROUGE1": scores["ROUGE-1"],

        "ROUGE2": scores["ROUGE-2"],

        "ROUGEL": scores["ROUGE-L"],

        "Execution_Time": execution_time
    })


advanced_results_df = pd.DataFrame(advanced_results)

print("Advanced model evaluation completed!")

display(advanced_results_df.head())

Evaluating DistilBART:   0%|          | 0/100 [00:00<?, ?it/s]

Advanced model evaluation completed!


,Article_ID,Reference,BART_Summary,ROUGE1,ROUGE2,ROUGEL,Execution_Time
0,0,Membership gives the ICC jurisdiction over all...,The Palestinian Authority becomes the 123rd m...,0.535211,0.376812,0.478873,12.908660
1,1,"Theia, a bully breed mix, was apparently hit b...",Theia is a one-year-old bully breed mix who w...,0.505263,0.215054,0.442105,19.519129
2,2,Mohammad Javad Zarif has spent more time with ...,Mohammad Javad Zarif received a hero's welcom...,0.282353,0.072289,0.211765,16.456579
3,3,17 Americans were exposed to the Ebola virus w...,The five were exposed to Ebola in Sierra Leon...,0.375000,0.153846,0.300000,8.317932
4,4,Student is no longer on Duke University campus...,A Duke student has admitted to hanging a noos...,0.683544,0.389610,0.430380,9.248693


In [11]:
advanced_performance = pd.DataFrame({

    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Execution Time"
    ],

    "DistilBART": [

        advanced_results_df["ROUGE1"].mean(),

        advanced_results_df["ROUGE2"].mean(),

        advanced_results_df["ROUGEL"].mean(),

        advanced_results_df["Execution_Time"].mean()
    ]
})

print("ADVANCED MODEL PERFORMANCE")

display(advanced_performance.round(4))

ADVANCED MODEL PERFORMANCE


,Metric,DistilBART
0,ROUGE-1,0.3619
1,ROUGE-2,0.1602
2,ROUGE-L,0.2730
3,Execution Time,13.5952


In [12]:
advanced_results_df.to_csv(
    "advanced_distilbart_results.csv",
    index=False
)

advanced_performance.to_csv(
    "advanced_distilbart_performance.csv",
    index=False
)

print("Advanced model results saved successfully!")

Advanced model results saved successfully!
